# RAG From Scratch (Parts 1–4), adapted for Anthropic + Voyage AI

Adapted from LangChain's `rag_from_scratch_1_to_4.ipynb`. Changes from the original:

- **LLM:** `ChatOpenAI` → `ChatAnthropic` (Claude)
- **Embeddings:** `OpenAIEmbeddings` → `VoyageAIEmbeddings`
- **Vector store:** `langchain_community` Chroma → `langchain-chroma`
- **Imports updated** for current LangChain (`langchain_text_splitters`, `langchain_core.prompts`, `langsmith` for the prompt hub)
- **`get_relevant_documents`** → `retriever.invoke` (old method is deprecated)
- **Index built once** and reused, so you don't re-embed (or duplicate chunks in Chroma) in every section

## Environment
### (1) Packages

In [ ]:
# Run once in your conda env (drop the ! if running in a terminal)
! pip install langchain langchain-community langchain-anthropic langchain-voyageai langchain-chroma langchain-text-splitters langsmith tiktoken beautifulsoup4 numpy python-dotenv

### (2) API keys

Put these in a `.env` file next to the notebook (and add `.env` to `.gitignore`):

```
ANTHROPIC_API_KEY=sk-ant-...
VOYAGE_API_KEY=pa-...
LANGCHAIN_API_KEY=lsv2_pt_...
LANGCHAIN_TRACING_V2=true
LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
LANGCHAIN_PROJECT=rag-from-scratch
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Silences WebBaseLoader's user-agent warning
os.environ.setdefault("USER_AGENT", "rag-from-scratch-tutorial")

for key in ["ANTHROPIC_API_KEY", "VOYAGE_API_KEY", "LANGCHAIN_API_KEY"]:
    print(f"{key}: {'set' if os.getenv(key) else 'MISSING'}")

### (3) Models
Defined once and reused below. Check Voyage's docs for current model names if this one errors.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_voyageai import VoyageAIEmbeddings

llm = ChatAnthropic(model="claude-haiku-5-5", temperature=0)
embd = VoyageAIEmbeddings(model="voyage-3.5-lite")

## Part 1: Overview
RAG quickstart

In [ ]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langsmith import Client

#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed (separate collection name so Part 2's index doesn't mix with this one)
quickstart_store = Chroma.from_documents(
    documents=splits, embedding=embd, collection_name="quickstart"
)
retriever = quickstart_store.as_retriever()

#### RETRIEVAL and GENERATION ####

# Prompt (pulled from the LangChain prompt hub via LangSmith)
prompt = Client().pull_prompt("rlm/rag-prompt")

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question
rag_chain.invoke("What is Task Decomposition?")

## Part 2: Indexing

In [ ]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

Count tokens (~4 chars / token).

Note: `cl100k_base` is OpenAI's tokenizer. Claude and Voyage use their own, so treat this as an approximation.
It's still fine for sizing chunks.

In [ ]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(string))

num_tokens_from_string(question, "cl100k_base")

### Text embedding models
Voyage embeds queries and documents slightly differently, so use `embed_documents` for the document.

In [ ]:
query_result = embd.embed_query(question)
document_result = embd.embed_documents([document])[0]
len(query_result)

Cosine similarity (1 = identical). Voyage embeddings are normalized, so this works the same way it did with OpenAI.

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

### Document loaders

In [ ]:
#### INDEXING ####

# Load blog
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

### Splitter

`RecursiveCharacterTextSplitter` tries to split on `["\n\n", "\n", " ", ""]` in order until chunks are small enough,
which keeps paragraphs, then sentences, then words together as long as possible.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50,
)
splits = text_splitter.split_documents(blog_docs)
len(splits)

### Vectorstores
This index is built once here and reused in Parts 3 and 4.

In [ ]:
vectorstore = Chroma.from_documents(
    documents=splits, embedding=embd, collection_name="blog_300"
)
retriever = vectorstore.as_retriever()

## Part 3: Retrieval
Reuses the index from Part 2 instead of re-embedding.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
docs = retriever.invoke("What is Task Decomposition?")
len(docs)

## Part 4: Generation

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

In [ ]:
# Chain
chain = prompt | llm

# Run
chain.invoke({"context": docs, "question": "What is Task Decomposition?"})

In [ ]:
# The hub prompt, for comparison
prompt_hub_rag = Client().pull_prompt("rlm/rag-prompt")
prompt_hub_rag

### RAG chains

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")